In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import h5py

import openquantum_sde
from openquantum_sde.io import load_trajectory, load_params
from openquantum_sde.utils import calculate_num_photons, calculate_num_photons_chunk


In [ ]:
# Lookup data folder
if "DATA" in os.environ:
    base_dir = Path(os.environ["DATA"]).expanduser()
else:
    base_dir = Path(".")  # current folder
PROJECT_NAME = "openquantum_sde"

In [ ]:
# Function to calculate number of photons
def calculate_num_photons_from_trajfiles(output_dir, chunk_size, nreps=1):
    '''
    Loads a random continuous chunk of trajectory nreps and calculate the average number of photons
    '''
    file_list = [
        Path(output_dir) / f"traj_CK_{i:04d}.h5"
        for i in range(1, 11)
    ]

    total_photons = 0.0
    total_norm = 0.0

    for _ in range(nreps):

        filename = np.random.choice(file_list)

        with h5py.File(filename, "r") as f:
            traj = f["traj"]

            start = np.random.randint(
                0, traj.shape[0] - chunk_size
            )

            chunk = traj[start:start + chunk_size]

            photons, norm = calculate_num_photons_chunk(chunk)

            total_photons += photons
            total_norm += norm

    return total_photons / total_norm

In [ ]:
# Load data
epsilon_array = list(range(2, 30 + 1)) #[10]
num_photons = [None] * len(epsilon_array)
for i, epsilon in enumerate(epsilon_array):
    SIM_NAME = "transmon_cavity_eps_" + str(int(epsilon)) + "/data"
    simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
    num_photons[i] = calculate_num_photons_from_trajfiles(simulation_dir, chunk_size = 5000, nreps=10)

In [ ]:
print(num_photons)
fig, ax = plt.subplots(figsize=(5, 3), dpi=120)
ax.plot(epsilon_array, num_photons, label=r'$\langle n_{ph} \rangle$', lw=1.0)
ax.set_xlabel(r'Drive $(\epsilon)$')
ax.set_ylabel(r'Average photon number')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Check size and shape of h5 file
SIM_NAME = "transmon_cavity_eps_" + str(int(10)) + "/data"
simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
fname = "traj_CK_0001.h5"
filename = simulation_dir / fname
with h5py.File(filename, "r") as f:
    traj = f["traj"]
    print("shape:", traj.shape)
    print("chunks:", traj.chunks)
    print("size GB:", traj.size * traj.dtype.itemsize / 1e9)